In [ ]:
# PyTorch functions/methods helpers

# 6.6.1
with tempfile.TemporaryDirectory() as tmpdir: # tempfile is a built-in Python standard-library module. TemporaryDirectory() creates a directory that cleans up when the `with` block ends
    path = Path(tmpdir) / "tensor.pt" # Path comes from Python's pathlib module. It represents a filesystem path and makes path manipulation convenient and cross-platform
    original = torch.arange(6, dtype=torch.float32).reshape(2, 3)
    torch.save(original, path) # Saves/serializes the object `original` to the file at `path`
    loaded = torch_load(path) # wrapper for torch.load(path), which loads/deserializes the PyTorch object stored at `path` and reconstructs it

# 6.6.2
with tempfile.TemporaryDirectory() as tmpdir:
    path = Path(tmpdir) / "model_state.pt"
    torch.save(model.state_dict(), path) # Store the model's state_dict (which contains its parameters/buffers) in the filepath

    restored = nn.Sequential(nn.Linear(3, 4), nn.ReLU(), nn.Linear(4,1))
    restored.load_state_dict(torch_load(path, map_location="cpu")) # Load the saved state_dict and copy its parameter values into restored
                                                                   # This does not run the params through the model. It takes the saved weights/biases and puts them inside the new restored model
                                                                   # In this case, we chose map_location="cpu", but there are otehr options (GPU if cuda, and MPS if Apple)
    after = restored(X).detach()

* Training changes model state over time. File I/O is how that state survives beyond the current notebook session.

* This chapter stays notebook-native, but it teaches the same save/load concepts needed for reliable experiments.

# How to use this notebook

* Run the notebook from top to bottom.

* Every code block is designed to be cloud-runnable and self-contained inside this notebook.

* The drills are intentionally small: predict the shape or behavior first, run the cell, then read the assertion as the contract you must understand.

# You are done when you can

- explain why saving values is different from saving architecture code
- save and load tensors in notebook-safe temporary paths
- save and restore a model with `state_dict`
- resume optimizer state from a checkpoint dictionary
- recognize architecture mismatch errors while loading

In [ ]:
import math
from pathlib import Path
import tempfile

import torch
from torch import nn

torch.manual_seed(0)
torch.set_printoptions(precision=4, sci_mode=False)

def shape(x):
    return tuple(x.shape)

def count_scalars(parameters):
    return sum(p.numel() for p in parameters)

def torch_load(path, map_location=None):
  try:
    return torch.load(path, map_location=map_location, weights_only=True)
  except TypeError:
    return torch.load(path, map_location=map_location)

# 6.6.0 The Problem This Notebook Solves

So far, every notebook cell has lived in memory. Once the kernel restarts, ordinary Python variables disappear.

That is fine for tiny drills, but learned model state should not vanish just because a session ends.

File I/O answers several different needs:

- save a **tensor** so it can be reused
- save **model weights** so predictions can be reproduced
- save **optimizer** state so training can resume with its momentum or other internal state
- save **metadata** so a checkpoint is interpretable later

The most important theoretical distinction is:

```text
architecture: the code that defines the computation
state: the tensor values currently inside that computation
checkpoint: state plus enough context to resume or interpret work
```

This notebook uses temporary paths so it remains cloud-safe. The concept transfers directly to persistent paths later.

# 6.6.1 Save and Load Tensors

The smallest save/load unit is a tensor. Saving a tensor writes its values, dtype, shape, and enough PyTorch metadata to reconstruct it.

This is not a model checkpoint yet. It is just serialization of one object.

Starting here keeps the mechanism visible before model state dictionaries add naming and architecture compatibility.

Temporary directories keep this notebook cloud-safe. The files exist only while the cell runs.

In [ ]:
with tempfile.TemporaryDirectory() as tmpdir: # tempfile is a built-in Python standard-library module. TemporaryDirectory() creates a directory that cleans up when the `with` block ends
    path = Path(tmpdir) / "tensor.pt" # Path comes from Python's pathlib module. It represents a filesystem path and makes path manipulation convenient and cross-platform
    original = torch.arange(6, dtype=torch.float32).reshape(2, 3)
    torch.save(original, path) # Saves/serializes the object `original` to the file at `path`
    loaded = torch_load(path) # wrapper for torch.load(path), which loads/deserializes the PyTorch object stored at `path` and reconstructs it

print("loaded tensor:")
print(loaded)

assert torch.equal(original, loaded)
assert shape(loaded) == (2, 3)

loaded tensor:
tensor([[0., 1., 2.],
        [3., 4., 5.]])


# 6.6.2 Save and Restore Model Weights

Saving a model's `state_dict` saves tensor state, not the architecture definition. That means loading has two steps:

```text
recreate the same architecture in code
load the saved tensor values into that architecture
```

The theory is that the learned function depends on both pieces:

- architecture decides how tensors are used
- state decides the learned values used by that architecture

If both match, the restored model should produce the same predictions for the same input.

The cell proves that by comparing predictions before and after loading.

In [ ]:
torch.manual_seed(0)
model = nn.Sequential(
    nn.Linear(3, 4),
    nn.ReLU(),
    nn.Linear(4,1)
)
X = torch.randn(5, 3)
before = model(X).detach()

with tempfile.TemporaryDirectory() as tmpdir:
    path = Path(tmpdir) / "model_state.pt"
    torch.save(model.state_dict(), path) # Store the model's state_dict (which contains its parameters/buffers) in the filepath

    restored = nn.Sequential(nn.Linear(3, 4), nn.ReLU(), nn.Linear(4,1))
    restored.load_state_dict(torch_load(path, map_location="cpu")) # Load the saved state_dict and copy its parameter values into restored
                                                                   # This does not run the params through the model. It takes the saved weights/biases and puts them inside the new restored model
                                                                   # In this case, we chose map_location="cpu", but there are otehr options (GPU if cuda, and MPS if Apple)
    after = restored(X).detach()

print("max prediction difference:", float((before - after).abs().max()))

assert torch.allclose(before, after)

max prediction difference: 0.0


# 6.6.3 Checkpoints Usually Need More Than Model Weights

A model `state_dict` is enough for inference. It is often not enough for training resume.

Optimizers can have their own state.
* Momentum is the simplest example: the optimizer remembers a **running update direction**.
* Adam-style optimizers remember even more (e.g. **running averages**).
* If you reload only model weights and create a fresh optimizer, you may not truly resume the same training process.
* For plain SGD without momentum or other stateful features, only keeping the param states are ok since the gradient is the only thing that the optim keeps, which transpires to the params after `optimizer.step()`.

A checkpoint is usually a dictionary with several pieces:

```text
model_state: learned model tensor values
optimizer_state: optimizer internal state
epoch or step: where training stopped
metadata: short context for humans and future code
```

This is still a notebook drill, not a persistent experiment artifact.

The purpose is to understand what belongs in a checkpoint before larger training runs depend on it.

In [ ]:
torch.manual_seed(1)
model = nn.Linear(2, 1)
optimizer = torch.optim.SGD(model.parameters(), lr=0.1, momentum=0.9)

X = torch.randn(4, 2)
y = torch.randn(4, 1)
loss = ((model(X) - y) ** 2).mean()
optimizer.zero_grad()
loss.backward()
optimizer.step()

with tempfile.TemporaryDirectory() as tmpdir:

    path = Path(tmpdir) / "checkpoint.pt"
    torch.save(
        {
            "model_state": model.state_dict(),
            "optimizer_state": optimizer.state_dict(),
            "epoch": 1,
            "description": "one synthethic regression step",
        },
        path,
    )

    loaded = torch_load(path, map_location="cpu")
    new_model = nn.Linear(2, 1)
    new_optimizer = torch.optim.SGD(model.parameters(), lr=0.1, momentum=0.9)

    new_model.load_state_dict(loaded["model_state"])
    new_optimizer.load_state_dict(loaded["optimizer_state"])

print("checkpoint keys:", sorted(loaded.keys()))
print("loaded epoch:", loaded["epoch"])

assert loaded["epoch"] == 1
assert "state" in loaded["optimizer_state"] # "state" as a key was already written into state_dict() when you ran "optimizer_state": optimizer.state_dict()

checkpoint keys: ['description', 'epoch', 'model_state', 'optimizer_state']
loaded epoch: 1


# 6.6.4 Break It Deliberately: Load Into the Wrong Architecture

A saved state dictionary is not magic. The receiving architecture must have compatible parameter names and shapes.

**If the saved layer has weight shape for 3 input features and the new layer expects 4 input features, PyTorch refuses to load it.**

That refusal is good. Silent loading would mean the model's learned state no longer matches the computation that uses it.

The theory-level mistake is mixing state from one function family with architecture code from another.

The shape mismatch is the mechanical evidence.

In [ ]:
source = nn.Linear(3, 1)

with tempfile.TemporaryDirectory() as tmpdir:
    path = Path(tmpdir) / "linear.pt"
    torch.save(source.state_dict(), path)

    wrong = nn.Linear(4, 1)
    try:
        wrong.load_state_dict(torch_load(path, map_location="cpu")) # Will not work since the saved params from the linear layer are part of layer shape (3, 1), not (4, 1), so params will not be compatible
    except RuntimeError as err:
        print(type(err).__name__)
        print(str(err).splitlines()[0])
    else:
        raise AssertionError("Loading should fail because the input feature count changed.")

RuntimeError
Error(s) in loading state_dict for Linear:


# 6.6 Checkpoint

Answer these before moving on.

You do not need a separate notes file for chapters; short answers in markdown cells or in your own study notes are enough.

1. What is the difference between architecture and state?
> * Architecture describes the structure of the model: which layers it has, how they're connected, and their dimensions (e.g. `Linear(3, 4)` → ReLU → `Linear(4, 1)`)
> * State is the current data/values inside that structure, especially learned parameters and persistent buffers
> * The optimizer also has its own state, which is separate from the model's state

2. What does a model `state_dict` save, and what does it not save?
> * A model `state_dict` saves the model's parameters and persistent buffers. It does not save the architecture or optimizer state
> * The optimizer has its own `optimizer.state_dict()`, which can be saved separately in the same checkpoint

3. Why did predictions match after loading into the same architecture?
> Because the values of params are also loaded, so same data -> same params -> same predictions/inference results

4. Why can optimizer state matter when resuming training?
> * Some optimizers, such as Adam or SGD with momentum, maintain additional state based on previous gradients
> * For example, Adam keeps running estimates related to the gradients and squared gradients. This optimizer state affects future parameter updates, so if I restore only the model parameters and create a fresh optimizer, training may proceed differently from where it left off

5. What does a size mismatch during loading usually mean?
> * Means your new layer likely has the wrong shape, since the params from the saved layers are not compatible to the new layer
> * For example, a saved `nn.Linear(3, 1)` has a weight of shape `(1, 3)`, while `nn.Linear(4, 1)` expects `(1, 4)`